# S5-01: MCP 기초 — FastMCP 설정과 첫 도구
**Skilljar L01-L03: Introducing MCP / MCP Clients / Project Setup**

## 학습 목표
- MCP의 3계층 아키텍처 (Host → Client → Server)를 이해한다
- FastMCP를 설치하고 기본 MCP 서버를 구현한다
- `@mcp.tool()` 데코레이터로 도구를 정의하고 테스트한다
- 건축공학 구조 계산 도구를 MCP로 구현한다

## 사전 준비
1. Python 3.11+ 설치
2. Node.js 설치 (MCP Inspector 실행용)
3. 이 노트북과 같은 폴더에 `.env` 파일 생성:
```
ANTHROPIC_API_KEY="sk-ant-api03-your-key-here"
```

In [ ]:
# 패키지 설치
%pip install "mcp[cli]" anthropic python-dotenv pydantic

In [ ]:
# 환경변수 로드
from dotenv import load_dotenv
load_dotenv()

## 1. MCP 핵심 개념 이해

MCP (Model Context Protocol)의 3계층 구조:

| 계층 | 역할 | 예시 |
| --- | --- | --- |
| **Host** | MCP 클라이언트를 생성·관리하는 AI 앱 | Claude Desktop, Claude Code |
| **Client** | 하나의 서버와 1:1 연결되는 커넥터 | Host 내부에서 자동 생성 |
| **Server** | Tools/Resources/Prompts를 외부에 제공 | `structural_server.py` |

MCP 서버가 제공하는 3가지 Primitive:

| Primitive | 용도 | 비유 |
| --- | --- | --- |
| **Tools** | 실행 가능한 함수 | 계산기, API 호출 |
| **Resources** | 읽기 전용 데이터 소스 | 파일, DB 레코드 |
| **Prompts** | 재사용 가능한 프롬프트 템플릿 | 검토 양식 |

## 2. 첫 MCP 서버 만들기

FastMCP는 데코레이터 기반으로 MCP 서버를 쉽게 구현할 수 있는 프레임워크이다.

> **참고**: MCP 서버는 별도 프로세스로 실행되므로, 노트북에서는 서버 코드를 **파일로 저장**한 후 Inspector나 클라이언트로 테스트한다.

In [ ]:
# 첫 MCP 서버를 파일로 저장
server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("hello-server")

@mcp.tool()
def greet(name: str) -> str:
    """사용자에게 인사합니다."""
    return f"안녕하세요, {name}님! MCP 서버에서 인사드립니다."

@mcp.tool()
def add(a: float, b: float) -> float:
    """두 수를 더합니다."""
    return a + b

@mcp.tool()
def multiply(a: float, b: float) -> float:
    """두 수를 곱합니다."""
    return a * b

if __name__ == "__main__":
    mcp.run()
'''

with open("hello_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("hello_server.py 저장 완료")
print("테스트 방법: 터미널에서 'mcp dev hello_server.py' 실행")

## 3. 도구 정의 패턴 이해

`@mcp.tool()` 데코레이터의 자동 변환 규칙:
- **함수명** → 도구명
- **타입 힌트** → JSON Schema (파라미터 정의)
- **docstring** → 도구 설명 (LLM이 도구 선택 시 참고)
- **기본값** → optional 파라미터

In [ ]:
# FastMCP가 생성하는 JSON Schema를 확인해보자
# (노트북에서 직접 서버 객체를 만들어 스키마만 확인)

from mcp.server.fastmcp import FastMCP
import json

test_mcp = FastMCP("test")

@test_mcp.tool()
def calculate_area(
    width: float,
    height: float,
    unit: str = "m"
) -> dict:
    """직사각형 면적을 계산합니다.

    Args:
        width: 너비 (기본 단위: m)
        height: 높이 (기본 단위: m)
        unit: 단위 (m, cm, mm)
    """
    area = width * height
    return {"area": area, "unit": f"{unit}²"}

# 도구 목록에서 스키마 확인
tools = test_mcp._tool_manager.list_tools()
for tool in tools:
    print(f"도구명: {tool.name}")
    print(f"설명: {tool.description}")
    print(f"입력 스키마:")
    print(json.dumps(tool.inputSchema, indent=2, ensure_ascii=False))

---
## 연습 1: 단위 변환 MCP 도구 만들기

### 문제
건축공학에서 자주 사용하는 단위 변환 도구를 MCP 서버로 구현하세요.

**요구사항:**
1. `convert_length(value, from_unit, to_unit)` — 길이 변환 (mm, cm, m)
2. `convert_force(value, from_unit, to_unit)` — 힘 변환 (N, kN, MN)
3. `convert_stress(value, from_unit, to_unit)` — 응력 변환 (Pa, kPa, MPa, GPa)

**기대 출력:**
```
convert_length(1500, "mm", "m") → {"value": 1.5, "from": "1500 mm", "to": "1.5 m"}
convert_force(2500, "kN", "N") → {"value": 2500000, "from": "2500 kN", "to": "2500000 N"}
convert_stress(24, "MPa", "kPa") → {"value": 24000, "from": "24 MPa", "to": "24000 kPa"}
```

In [ ]:
# TODO: 아래 코드를 완성하세요

from mcp.server.fastmcp import FastMCP

unit_mcp = FastMCP("unit-converter")

# 길이 변환 계수 (모든 단위를 mm 기준으로)
LENGTH_TO_MM = {"mm": 1, "cm": 10, "m": 1000}

@unit_mcp.tool()
def convert_length(value: float, from_unit: str, to_unit: str) -> dict:
    """길이 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 원래 단위 (mm, cm, m)
        to_unit: 변환할 단위 (mm, cm, m)
    """
    # TODO: 변환 로직 구현
    pass

# TODO: convert_force, convert_stress도 구현하세요

In [ ]:
# === 솔루션 ===

from mcp.server.fastmcp import FastMCP

unit_mcp = FastMCP("unit-converter")

LENGTH_TO_MM = {"mm": 1, "cm": 10, "m": 1000}
FORCE_TO_N = {"N": 1, "kN": 1000, "MN": 1e6}
STRESS_TO_PA = {"Pa": 1, "kPa": 1000, "MPa": 1e6, "GPa": 1e9}

@unit_mcp.tool()
def convert_length(value: float, from_unit: str, to_unit: str) -> dict:
    """길이 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 원래 단위 (mm, cm, m)
        to_unit: 변환할 단위 (mm, cm, m)
    """
    if from_unit not in LENGTH_TO_MM or to_unit not in LENGTH_TO_MM:
        return {"error": f"지원 단위: {list(LENGTH_TO_MM.keys())}"}
    mm_value = value * LENGTH_TO_MM[from_unit]
    result = mm_value / LENGTH_TO_MM[to_unit]
    return {"value": result, "from": f"{value} {from_unit}", "to": f"{result} {to_unit}"}

@unit_mcp.tool()
def convert_force(value: float, from_unit: str, to_unit: str) -> dict:
    """힘 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 원래 단위 (N, kN, MN)
        to_unit: 변환할 단위 (N, kN, MN)
    """
    if from_unit not in FORCE_TO_N or to_unit not in FORCE_TO_N:
        return {"error": f"지원 단위: {list(FORCE_TO_N.keys())}"}
    n_value = value * FORCE_TO_N[from_unit]
    result = n_value / FORCE_TO_N[to_unit]
    return {"value": result, "from": f"{value} {from_unit}", "to": f"{result} {to_unit}"}

@unit_mcp.tool()
def convert_stress(value: float, from_unit: str, to_unit: str) -> dict:
    """응력 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 원래 단위 (Pa, kPa, MPa, GPa)
        to_unit: 변환할 단위 (Pa, kPa, MPa, GPa)
    """
    if from_unit not in STRESS_TO_PA or to_unit not in STRESS_TO_PA:
        return {"error": f"지원 단위: {list(STRESS_TO_PA.keys())}"}
    pa_value = value * STRESS_TO_PA[from_unit]
    result = pa_value / STRESS_TO_PA[to_unit]
    return {"value": result, "from": f"{value} {from_unit}", "to": f"{result} {to_unit}"}

# 테스트
print("=== 변환 테스트 ===")
print(convert_length(1500, "mm", "m"))
print(convert_force(2500, "kN", "N"))
print(convert_stress(24, "MPa", "kPa"))

In [ ]:
# 검증 함수
def verify_exercise1():
    tests = [
        (convert_length(1500, "mm", "m"), 1.5, "길이 변환"),
        (convert_length(2.5, "m", "cm"), 250, "길이 변환 2"),
        (convert_force(2500, "kN", "N"), 2500000, "힘 변환"),
        (convert_stress(24, "MPa", "kPa"), 24000, "응력 변환"),
        (convert_stress(200, "GPa", "MPa"), 200000, "응력 변환 2"),
    ]

    passed = 0
    for result, expected, name in tests:
        actual = result["value"]
        if abs(actual - expected) < 0.01:
            print(f"  PASS: {name} — {actual}")
            passed += 1
        else:
            print(f"  FAIL: {name} — expected {expected}, got {actual}")

    print(f"\n결과: {passed}/{len(tests)} 통과")
    return passed == len(tests)

verify_exercise1()

---
## 연습 2: Pydantic 모델을 활용한 도구 정의

### 문제
Pydantic `BaseModel`을 사용하여 복잡한 입력을 받는 MCP 도구를 정의하세요.

**요구사항:**
1. `RectSection` 모델 정의 (b: 폭 mm, h: 높이 mm, cover: 피복 두께 mm=40)
2. `section_properties(section: RectSection)` 도구 — 단면 성질 계산
   - 단면적 Ag = b * h
   - 유효 깊이 d = h - cover
   - 단면2차모멘트 Ig = b * h³ / 12
   - 단면계수 S = Ig / (h/2)

**기대 출력:**
```python
section_properties(RectSection(b=350, h=600))
# → {"Ag_mm2": 210000, "d_mm": 560, "Ig_mm4": 6300000000, "S_mm3": 21000000}
```

In [ ]:
# TODO: 아래 코드를 완성하세요

from pydantic import BaseModel, Field
from mcp.server.fastmcp import FastMCP

section_mcp = FastMCP("section-server")

class RectSection(BaseModel):
    """직사각형 단면 정보"""
    # TODO: 필드 정의
    pass

@section_mcp.tool()
def section_properties(section: RectSection) -> dict:
    """직사각형 단면의 기본 성질을 계산합니다."""
    # TODO: 계산 구현
    pass

In [ ]:
# === 솔루션 ===

from pydantic import BaseModel, Field
from mcp.server.fastmcp import FastMCP

section_mcp = FastMCP("section-server")

class RectSection(BaseModel):
    """직사각형 단면 정보"""
    b: float = Field(description="단면 폭 (mm)")
    h: float = Field(description="단면 높이 (mm)")
    cover: float = Field(default=40, description="피복 두께 (mm)")

@section_mcp.tool()
def section_properties(section: RectSection) -> dict:
    """직사각형 단면의 기본 성질을 계산합니다.

    Args:
        section: 직사각형 단면 정보 (b, h, cover)
    """
    b, h, cover = section.b, section.h, section.cover

    Ag = b * h
    d = h - cover
    Ig = b * h**3 / 12
    S = Ig / (h / 2)

    return {
        "Ag_mm2": round(Ag, 1),
        "d_mm": round(d, 1),
        "Ig_mm4": round(Ig, 1),
        "S_mm3": round(S, 1)
    }

# 테스트
result = section_properties(RectSection(b=350, h=600))
print(result)

In [ ]:
# 검증 함수
def verify_exercise2():
    result = section_properties(RectSection(b=350, h=600))

    expected = {
        "Ag_mm2": 210000,
        "d_mm": 560,
        "Ig_mm4": 6300000000,
        "S_mm3": 21000000
    }

    passed = 0
    for key, exp_val in expected.items():
        actual = result.get(key)
        if actual is not None and abs(actual - exp_val) < 1:
            print(f"  PASS: {key} = {actual}")
            passed += 1
        else:
            print(f"  FAIL: {key} — expected {exp_val}, got {actual}")

    # cover 기본값 테스트
    result2 = section_properties(RectSection(b=500, h=500, cover=50))
    if abs(result2["d_mm"] - 450) < 1:
        print(f"  PASS: cover=50 → d=450")
        passed += 1
    else:
        print(f"  FAIL: cover=50 → d should be 450, got {result2['d_mm']}")

    print(f"\n결과: {passed}/{len(expected)+1} 통과")
    return passed == len(expected) + 1

verify_exercise2()

---
## 연습 3: 서버 파일 작성 + Inspector 테스트 준비

### 문제
아래 요구사항에 맞는 MCP 서버를 `calc_server.py` 파일로 작성하세요.

**요구사항:**
1. 서버 이름: `"engineering-calc"`
2. 도구 3개:
   - `circle_area(radius: float)` — 원의 면적 (pi * r²)
   - `triangle_area(base: float, height: float)` — 삼각형 면적 (0.5 * b * h)
   - `trapezoid_area(a: float, b: float, height: float)` — 사다리꼴 면적 ((a+b)/2 * h)
3. 모든 도구에 한국어 docstring 작성
4. `mcp.run()` 포함

**테스트 방법:**
```bash
mcp dev calc_server.py
```
Inspector에서 각 도구를 호출하여 결과를 확인하세요.

In [ ]:
# TODO: calc_server.py 파일 내용을 작성하세요

server_code = '''
# TODO: 여기에 MCP 서버 코드를 작성하세요
'''

with open("calc_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("calc_server.py 저장 완료")
print("테스트: 터미널에서 'mcp dev calc_server.py' 실행")

In [ ]:
# === 솔루션 ===
import math

server_code = '''
import math
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("engineering-calc")

@mcp.tool()
def circle_area(radius: float) -> dict:
    """원의 면적을 계산합니다.

    Args:
        radius: 반지름
    """
    area = math.pi * radius ** 2
    return {"shape": "circle", "radius": radius, "area": round(area, 4)}

@mcp.tool()
def triangle_area(base: float, height: float) -> dict:
    """삼각형의 면적을 계산합니다.

    Args:
        base: 밑변 길이
        height: 높이
    """
    area = 0.5 * base * height
    return {"shape": "triangle", "base": base, "height": height, "area": round(area, 4)}

@mcp.tool()
def trapezoid_area(a: float, b: float, height: float) -> dict:
    """사다리꼴의 면적을 계산합니다.

    Args:
        a: 윗변 길이
        b: 아랫변 길이
        height: 높이
    """
    area = (a + b) / 2 * height
    return {"shape": "trapezoid", "a": a, "b": b, "height": height, "area": round(area, 4)}

if __name__ == "__main__":
    mcp.run()
'''

with open("calc_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("calc_server.py 저장 완료")

# 로컬 테스트
print("\n=== 로컬 테스트 ===")
print(f"원 (r=5): 면적 = {round(math.pi * 25, 4)}")
print(f"삼각형 (b=10, h=6): 면적 = {0.5 * 10 * 6}")
print(f"사다리꼴 (a=4, b=8, h=5): 면적 = {(4+8)/2 * 5}")

In [ ]:
# 검증 함수 — 파일 존재 및 내용 확인
import os

def verify_exercise3():
    passed = 0

    # 파일 존재 확인
    if os.path.exists("calc_server.py"):
        print("  PASS: calc_server.py 파일 존재")
        passed += 1
    else:
        print("  FAIL: calc_server.py 파일이 없습니다")
        return False

    with open("calc_server.py", "r", encoding="utf-8") as f:
        content = f.read()

    checks = [
        ("FastMCP" in content, "FastMCP 임포트"),
        ("@mcp.tool()" in content, "@mcp.tool() 데코레이터"),
        ("circle_area" in content, "circle_area 함수"),
        ("triangle_area" in content, "triangle_area 함수"),
        ("trapezoid_area" in content, "trapezoid_area 함수"),
        ("mcp.run()" in content, "mcp.run() 호출"),
        ('"""' in content or "'''" in content, "docstring 존재"),
    ]

    for check, name in checks:
        if check:
            print(f"  PASS: {name}")
            passed += 1
        else:
            print(f"  FAIL: {name}")

    print(f"\n결과: {passed}/{len(checks)+1} 통과")
    return passed == len(checks) + 1

verify_exercise3()

---
## 건축공학 실습: RC 보 설계 도구 MCP 서버

### 과제
RC 보 설계에 필요한 3가지 계산 도구를 MCP 서버 파일 (`beam_server.py`)로 작성하세요.

**요구 도구:**
1. `calc_moment_capacity(b, d, fck, fy, As)` — 공칭 휨모멘트 Mn 및 설계 휨모멘트 phi_Mn 계산
   - a = As * fy / (0.85 * fck * b)
   - Mn = As * fy * (d - a/2) / 1e6 (kN·m)
   - phi_Mn = 0.85 * Mn

2. `calc_shear_capacity(b, d, fck)` — 콘크리트 전단강도 Vc 계산
   - Vc = (1/6) * sqrt(fck) * b * d / 1000 (kN)
   - phi_Vc = 0.75 * Vc

3. `calc_min_rebar(b, d, fck, fy)` — 최소 철근량 산정
   - rho_min = max(0.25 * sqrt(fck) / fy, 1.4 / fy)
   - As_min = rho_min * b * d

**기대 출력 (b=350, d=560, fck=27, fy=400, As=2534):**
```
calc_moment_capacity: Mn ≈ 498 kN·m, phi_Mn ≈ 423 kN·m
calc_shear_capacity: Vc ≈ 160 kN, phi_Vc ≈ 120 kN
calc_min_rebar: rho_min ≈ 0.0035, As_min ≈ 686 mm²
```

In [ ]:
# TODO: beam_server.py 코드를 완성하세요

import math

server_code = '''
import math
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("beam-design-server")

# TODO: 3가지 도구를 구현하세요

if __name__ == "__main__":
    mcp.run()
'''

with open("beam_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("beam_server.py 저장 완료")

In [ ]:
# === 솔루션 ===
import math

server_code = '''
import math
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("beam-design-server")

@mcp.tool()
def calc_moment_capacity(
    b: float, d: float, fck: float, fy: float, As: float
) -> dict:
    """RC 보의 공칭 휨모멘트 강도를 계산합니다 (KDS 14 20 20).

    Args:
        b: 단면 폭 (mm)
        d: 유효 깊이 (mm)
        fck: 콘크리트 설계기준강도 (MPa)
        fy: 철근 항복강도 (MPa)
        As: 인장 철근량 (mm2)
    """
    a = (As * fy) / (0.85 * fck * b)
    Mn = As * fy * (d - a / 2) / 1e6
    phi_Mn = 0.85 * Mn
    return {
        "a_mm": round(a, 1),
        "Mn_kNm": round(Mn, 1),
        "phi_Mn_kNm": round(phi_Mn, 1),
        "reference": "KDS 14 20 20, 4.2.3"
    }

@mcp.tool()
def calc_shear_capacity(b: float, d: float, fck: float) -> dict:
    """RC 보의 콘크리트 전단강도를 계산합니다 (KDS 14 20 22).

    Args:
        b: 단면 폭 (mm)
        d: 유효 깊이 (mm)
        fck: 콘크리트 설계기준강도 (MPa)
    """
    Vc = (1/6) * math.sqrt(fck) * b * d / 1000
    phi_Vc = 0.75 * Vc
    return {
        "Vc_kN": round(Vc, 1),
        "phi_Vc_kN": round(phi_Vc, 1),
        "reference": "KDS 14 20 22, 5.3.1"
    }

@mcp.tool()
def calc_min_rebar(b: float, d: float, fck: float, fy: float) -> dict:
    """RC 보의 최소 철근량을 산정합니다 (KDS 14 20 20).

    Args:
        b: 단면 폭 (mm)
        d: 유효 깊이 (mm)
        fck: 콘크리트 설계기준강도 (MPa)
        fy: 철근 항복강도 (MPa)
    """
    rho_min = max(0.25 * math.sqrt(fck) / fy, 1.4 / fy)
    As_min = rho_min * b * d
    return {
        "rho_min": round(rho_min, 5),
        "As_min_mm2": round(As_min, 1),
        "reference": "KDS 14 20 20"
    }

if __name__ == "__main__":
    mcp.run()
'''

with open("beam_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("beam_server.py 저장 완료")

# 로컬 테스트 (함수 직접 호출)
b, d, fck, fy, As = 350, 560, 27, 400, 2534

a = (As * fy) / (0.85 * fck * b)
Mn = As * fy * (d - a / 2) / 1e6
phi_Mn = 0.85 * Mn
print(f"\n=== 휨강도 ===")
print(f"a = {a:.1f} mm, Mn = {Mn:.1f} kN·m, phi_Mn = {phi_Mn:.1f} kN·m")

Vc = (1/6) * math.sqrt(fck) * b * d / 1000
phi_Vc = 0.75 * Vc
print(f"\n=== 전단강도 ===")
print(f"Vc = {Vc:.1f} kN, phi_Vc = {phi_Vc:.1f} kN")

rho_min = max(0.25 * math.sqrt(fck) / fy, 1.4 / fy)
As_min = rho_min * b * d
print(f"\n=== 최소 철근량 ===")
print(f"rho_min = {rho_min:.5f}, As_min = {As_min:.1f} mm²")

In [ ]:
# 검증 함수
def verify_structural():
    import os

    if not os.path.exists("beam_server.py"):
        print("  FAIL: beam_server.py 파일이 없습니다")
        return False

    with open("beam_server.py", "r", encoding="utf-8") as f:
        content = f.read()

    checks = [
        ("FastMCP" in content, "FastMCP 사용"),
        ("calc_moment_capacity" in content, "휨강도 도구"),
        ("calc_shear_capacity" in content, "전단강도 도구"),
        ("calc_min_rebar" in content, "최소 철근량 도구"),
        ("KDS" in content, "KDS 기준 참조"),
        ("mcp.run()" in content, "서버 실행 코드"),
    ]

    passed = sum(1 for c, _ in checks if c)
    for check, name in checks:
        print(f"  {'PASS' if check else 'FAIL'}: {name}")

    print(f"\n결과: {passed}/{len(checks)} 통과")
    return passed == len(checks)

verify_structural()